# Logistic Regression Classifier training
- This is an auto-generated notebook.
- To reproduce these results, attach this notebook to a cluster with runtime version **15.4.x-cpu-ml-scala2.12**, and rerun it.
- Compare trials in the [MLflow experiment](#mlflow/experiments/1125739752927671).
- Clone this notebook into your project folder by selecting **File > Clone** in the notebook toolbar.

## Load Data

In [1]:
import sys
import warnings
import mlflow

sys.path.append("..")
from utils import get_table

warnings.filterwarnings("ignore")
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/maicolnicolini96@gmail.com/bet_analytics_experiments/local_training")

/Users/maicolnicolini/Desktop/Code/bet-master-analytics/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/maicolnicolini/Desktop/Code/bet-master-analytics/venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with critical bug fixes on a best-effort basis, but not
    with any other fixes or features. Please upgrade your Python version,
    and then update google-auth.
    
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/maicolnicolini/Desktop/Code/bet-master-analytics/venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with criti

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3256615592902883', creation_time=1769270327805, experiment_id='3256615592902883', last_update_time=1769271576157, lifecycle_stage='active', name='/Users/maicolnicolini96@gmail.com/bet_analytics_experiments/local_training', tags={'mlflow.experiment.sourceName': '/Users/maicolnicolini96@gmail.com/bet_analytics_experiments/local_training',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'maicolnicolini96@gmail.com',
 'mlflow.ownerId': '4218937150946106'}>

In [2]:
target_col = "win_1x"
time_col = "time"
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)

df_loaded.columns = [x.replace(".", "_") for x in df_loaded.columns]

df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Define proportions
train_prop = 0.8
validate_prop = 0.1
test_prop = 0.1

# Compute split indices
n = len(df_loaded)
train_end = int(n * train_prop)
validate_end = train_end + int(n * validate_prop)

# Create the split column
df_loaded['_automl_split_col_0000'] = ['train'] * train_end + ['validate'] * (validate_end - train_end) + ['test'] * (n - validate_end)

# Preview data
display(df_loaded.head(5))

,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under05HT,underOver_flashback_over05HT,underOver_flashback_under15,underOver_flashback_over15,underOver_flashback_under25,underOver_flashback_over25,underOver_flashback_under35,underOver_flashback_over35,win_1x,_automl_split_col_0000
0,2025-01-09 00:15:00+00:00,63.4,26.0,10.6,89.4,36.6,74.0,40.7,49.2,10.1,...,29.7,70.3,26.7,73.3,51.6,48.4,72.0,28.0,True,train
1,2025-01-09 01:30:00+00:00,57.9,26.5,15.6,84.4,42.1,73.5,42.2,30.3,27.5,...,31.7,68.3,27.7,72.3,51.3,48.7,74.1,25.9,True,train
2,2025-01-10 00:00:00+00:00,29.1,26.9,44.0,56.0,70.9,73.1,27.7,43.9,28.4,...,37.9,62.1,35.7,64.3,62.4,37.6,82.3,17.7,True,train
3,2025-01-10 02:30:00+00:00,53.8,26.0,20.2,79.8,46.2,74.0,43.9,40.8,15.3,...,35.5,64.5,33.6,66.4,59.0,41.0,79.5,20.5,True,train
4,2025-01-10 02:30:00+00:00,42.2,31.1,26.7,73.3,57.8,68.9,30.8,39.5,29.7,...,29.5,70.5,23.8,76.2,46.2,53.8,71.0,29.0,True,train


### Select supported columns
Select only the columns that are supported. This allows us to train a model that can predict on a dataset that has extra columns that are not used in training.
`["team_corner", "team_goalHt", "team_goal"]` are dropped in the pipelines. See the Alerts tab of the AutoML Experiment page for details on why these columns are dropped.

In [3]:
from databricks.automl_runtime.sklearn.column_selector import ColumnSelector

supported_cols = ["goalNoGoal_stats_avgGoalTakenAway", "underOver_quote_currentO", "goalNoGoal_chance_goal", "chance1x2_chance_p2", "chance1x2_chance_pHtx", "chance1x2_chance_p2Ht2x", "chance1x2_comparison_affini", "underOver_chance_over35", "underOver_chance_under15", "chance1x2_flashback_px", "goalNoGoal_multigoal_m35", "underOver_bookkeeping_u", "goalNoGoal_multigoal_m14", "chance1x2_chance_pHt1", "goalNoGoal_chance_goalAway", "chance1x2_quote_real2", "goalNoGoal_chance_even", "chance1x2_quote_diffInitialCurrx", "chance1x2_chance_p2Ht12", "goalNoGoal_chance_goalHome", "underOver_chance_over25", "chance1x2_chance_p1x", "chance1x2_quote_diffInitialCurr1", "chance1x2_quote_diffRealCurr1", "goalNoGoal_quote_realGG", "chance1x2_flashback_pHt1", "underOver_chance_over15HT", "chance1x2_quote_diffRealCurr2", "goalNoGoal_comparison_affini", "goalNoGoal_multigoal_m24Home", "evaluation_valUnderOver", "underOver_flashback_under35", "chance1x2_bookkeeping_status", "goalNoGoal_bookkeeping_actual", "goalNoGoal_multigoal_m24Away", "chance1x2_flashback_pHtx", "chance1x2_bookkeeping_p1", "chance1x2_quote_initial1", "goalNoGoal_chance_noGoal", "goalNoGoal_bookkeeping_arrow", "underOver_chance_under05HT", "underOver_quote_realU", "underOver_chance_over052HT", "chance1x2_chance_px", "underOver_bookkeeping_status", "evaluation_valScala", "underOver_chance_over05HT", "underOver_quote_diffRealCurrU", "underOver_quote_realO", "underOver_comparison_flashback", "goalNoGoal_stats_avgGoalHome", "chance1x2_chance_p2Ht1x", "underOver_comparison_affini", "underOver_chance_under15HT", "chance1x2_flashback_pHt2", "goalNoGoal_multigoal_m13", "underOver_quote_diffInitialCurrU", "goalNoGoal_quote_diffRealCurrGG", "goalNoGoal_stats_avgGoalTakenHome", "goalNoGoal_quote_realNG", "goalNoGoal_quote_diffInitialCurrGG", "chance1x2_chance_p1", "underOver_quote_diffInitialCurrO", "chance1x2_quote_initial2", "chance1x2_quote_realx", "goalNoGoal_flashback_noGoal", "underOver_chance_over15", "goalNoGoal_bookkeeping_status", "chance1x2_quote_current1", "underOver_chance_under45", "chance1x2_quote_initialx", "goalNoGoal_bookkeeping_gg", "underOver_chance_over45", "chance1x2_comparison_flashback", "chance1x2_flashback_p2", "chance1x2_bookkeeping_px", "goalNoGoal_multigoal_m13Away", "chance1x2_chance_p2x", "chance1x2_quote_currentx", "chance1x2_bookkeeping_actual", "underOver_flashback_over25", "goalNoGoal_quote_currentNG", "chance1x2_quote_diffRealCurrx", "underOver_chance_under052HT", "underOver_bookkeeping_o", "underOver_chance_under35", "goalNoGoal_bookkeeping_avg", "chance1x2_quote_current2", "goalNoGoal_flashback_goal", "underOver_flashback_over05HT", "underOver_flashback_under05HT", "evaluation_valMetrica", "underOver_flashback_over15", "chance1x2_quote_real1", "goalNoGoal_quote_initialNG", "underOver_bookkeeping_actual", "goalNoGoal_quote_initialGG", "chance1x2_chance_p12", "underOver_quote_currentU", "underOver_flashback_under25", "evaluation_val1x2", "goalNoGoal_stats_avgGoalAway", "goalNoGoal_flashback_m35", "goalNoGoal_flashback_m24", "underOver_quote_initialU", "goalNoGoal_quote_diffRealCurrNG", "goalNoGoal_multigoal_m24", "underOver_bookkeeping_arrow", "underOver_bookkeeping_avg", "goalNoGoal_chance_odd", "chance1x2_flashback_p1", "goalNoGoal_flashback_m13", "underOver_flashback_over35", "chance1x2_bookkeeping_arrow", "goalNoGoal_multigoal_m13Home", "time", "chance1x2_bookkeeping_p2", "underOver_quote_diffRealCurrO", "underOver_chance_under25", "goalNoGoal_quote_diffInitialCurrNG", "goalNoGoal_comparison_flashback", "underOver_quote_initialO", "chance1x2_chance_pHt2", "goalNoGoal_quote_currentGG", "chance1x2_bookkeeping_avg", "chance1x2_quote_diffInitialCurr2", "goalNoGoal_bookkeeping_ng", "underOver_flashback_under15"]
col_selector = ColumnSelector(supported_cols)

## Preprocessors

### Datetime Preprocessor
For each datetime column, extract relevant information from the date:
- Unix timestamp
- whether the date is a weekend
- whether the date is a holiday

Additionally, extract extra information from columns with timestamps:
- hour of the day (one-hot encoded)

For cyclic features, plot the values along a unit circle to encode temporal proximity:
- hour of the day
- hours since the beginning of the week
- hours since the beginning of the month
- hours since the beginning of the year

In [4]:
from pandas import Timestamp
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from databricks.automl_runtime.sklearn import DatetimeImputer
from databricks.automl_runtime.sklearn import OneHotEncoder
from databricks.automl_runtime.sklearn import TimestampTransformer
from sklearn.preprocessing import StandardScaler

imputers = {
  "time": DatetimeImputer(),
}

datetime_transformers = []

for col in ["time"]:
    ohe_transformer = ColumnTransformer(
        [("ohe", OneHotEncoder(sparse=False, handle_unknown="indicator"), [TimestampTransformer.HOUR_COLUMN_INDEX])],
        remainder="passthrough")
    timestamp_preprocessor = Pipeline([
        (f"impute_{col}", imputers[col]),
        (f"transform_{col}", TimestampTransformer()),
        (f"onehot_encode_{col}", ohe_transformer),
        (f"standardize_{col}", StandardScaler()),
    ])
    datetime_transformers.append((f"timestamp_{col}", timestamp_preprocessor, [col]))

### Boolean columns
For each column, impute missing values and then convert into ones and zeros.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder


bool_imputers = []

bool_pipeline = Pipeline(steps=[
    ("cast_type", FunctionTransformer(lambda df: df.astype(object))),
    ("imputers", ColumnTransformer(bool_imputers, remainder="passthrough")),
    ("onehot", SklearnOneHotEncoder(handle_unknown="ignore", drop="first")),
])

bool_transformers = [("boolean", bool_pipeline, ["chance1x2_bookkeeping_arrow", "goalNoGoal_bookkeeping_arrow", "underOver_bookkeeping_arrow"])]

### Numerical columns

Missing values for numerical columns are imputed with mean by default.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

num_imputers = []
num_imputers.append(("impute_mean", SimpleImputer(), ["chance1x2_bookkeeping_actual", "chance1x2_bookkeeping_avg", "chance1x2_bookkeeping_p1", "chance1x2_bookkeeping_p2", "chance1x2_bookkeeping_px", "chance1x2_bookkeeping_status", "chance1x2_chance_p1", "chance1x2_chance_p12", "chance1x2_chance_p1x", "chance1x2_chance_p2", "chance1x2_chance_p2Ht12", "chance1x2_chance_p2Ht1x", "chance1x2_chance_p2Ht2x", "chance1x2_chance_p2x", "chance1x2_chance_pHt1", "chance1x2_chance_pHt2", "chance1x2_chance_pHtx", "chance1x2_chance_px", "chance1x2_comparison_affini", "chance1x2_comparison_flashback", "chance1x2_flashback_p1", "chance1x2_flashback_p2", "chance1x2_flashback_pHt1", "chance1x2_flashback_pHt2", "chance1x2_flashback_pHtx", "chance1x2_flashback_px", "chance1x2_quote_current1", "chance1x2_quote_current2", "chance1x2_quote_currentx", "chance1x2_quote_diffInitialCurr1", "chance1x2_quote_diffInitialCurr2", "chance1x2_quote_diffInitialCurrx", "chance1x2_quote_diffRealCurr1", "chance1x2_quote_diffRealCurr2", "chance1x2_quote_diffRealCurrx", "chance1x2_quote_initial1", "chance1x2_quote_initial2", "chance1x2_quote_initialx", "chance1x2_quote_real1", "chance1x2_quote_real2", "chance1x2_quote_realx", "evaluation_val1x2", "evaluation_valMetrica", "evaluation_valScala", "evaluation_valUnderOver", "goalNoGoal_bookkeeping_actual", "goalNoGoal_bookkeeping_avg", "goalNoGoal_bookkeeping_gg", "goalNoGoal_bookkeeping_ng", "goalNoGoal_bookkeeping_status", "goalNoGoal_chance_even", "goalNoGoal_chance_goal", "goalNoGoal_chance_goalAway", "goalNoGoal_chance_goalHome", "goalNoGoal_chance_noGoal", "goalNoGoal_chance_odd", "goalNoGoal_comparison_affini", "goalNoGoal_comparison_flashback", "goalNoGoal_flashback_goal", "goalNoGoal_flashback_m13", "goalNoGoal_flashback_m24", "goalNoGoal_flashback_m35", "goalNoGoal_flashback_noGoal", "goalNoGoal_multigoal_m13", "goalNoGoal_multigoal_m13Away", "goalNoGoal_multigoal_m13Home", "goalNoGoal_multigoal_m14", "goalNoGoal_multigoal_m24", "goalNoGoal_multigoal_m24Away", "goalNoGoal_multigoal_m24Home", "goalNoGoal_multigoal_m35", "goalNoGoal_quote_currentGG", "goalNoGoal_quote_currentNG", "goalNoGoal_quote_diffInitialCurrGG", "goalNoGoal_quote_diffInitialCurrNG", "goalNoGoal_quote_diffRealCurrGG", "goalNoGoal_quote_diffRealCurrNG", "goalNoGoal_quote_initialGG", "goalNoGoal_quote_initialNG", "goalNoGoal_quote_realGG", "goalNoGoal_quote_realNG", "goalNoGoal_stats_avgGoalAway", "goalNoGoal_stats_avgGoalHome", "goalNoGoal_stats_avgGoalTakenAway", "goalNoGoal_stats_avgGoalTakenHome", "underOver_bookkeeping_actual", "underOver_bookkeeping_avg", "underOver_bookkeeping_o", "underOver_bookkeeping_status", "underOver_bookkeeping_u", "underOver_chance_over052HT", "underOver_chance_over05HT", "underOver_chance_over15", "underOver_chance_over15HT", "underOver_chance_over25", "underOver_chance_over35", "underOver_chance_over45", "underOver_chance_under052HT", "underOver_chance_under05HT", "underOver_chance_under15", "underOver_chance_under15HT", "underOver_chance_under25", "underOver_chance_under35", "underOver_chance_under45", "underOver_comparison_affini", "underOver_comparison_flashback", "underOver_flashback_over05HT", "underOver_flashback_over15", "underOver_flashback_over25", "underOver_flashback_over35", "underOver_flashback_under05HT", "underOver_flashback_under15", "underOver_flashback_under25", "underOver_flashback_under35", "underOver_quote_currentO", "underOver_quote_currentU", "underOver_quote_diffInitialCurrO", "underOver_quote_diffInitialCurrU", "underOver_quote_diffRealCurrO", "underOver_quote_diffRealCurrU", "underOver_quote_initialO", "underOver_quote_initialU", "underOver_quote_realO", "underOver_quote_realU"]))

numerical_pipeline = Pipeline(steps=[
    ("converter", FunctionTransformer(lambda df: df.apply(pd.to_numeric, errors='coerce'))),
    ("imputers", ColumnTransformer(num_imputers)),
    ("standardizer", StandardScaler()),
])

numerical_transformers = [("numerical", numerical_pipeline, ["goalNoGoal_stats_avgGoalTakenAway", "underOver_quote_currentO", "goalNoGoal_chance_goal", "chance1x2_chance_p2", "chance1x2_chance_pHtx", "chance1x2_chance_p2Ht2x", "chance1x2_comparison_affini", "underOver_chance_over35", "underOver_chance_under15", "chance1x2_flashback_px", "goalNoGoal_multigoal_m35", "underOver_bookkeeping_u", "goalNoGoal_multigoal_m14", "chance1x2_chance_pHt1", "goalNoGoal_chance_goalAway", "chance1x2_quote_real2", "goalNoGoal_chance_even", "chance1x2_quote_diffInitialCurrx", "chance1x2_chance_p2Ht12", "goalNoGoal_chance_goalHome", "underOver_chance_over25", "chance1x2_chance_p1x", "chance1x2_quote_diffInitialCurr1", "chance1x2_quote_diffRealCurr1", "goalNoGoal_quote_realGG", "chance1x2_flashback_pHt1", "underOver_chance_over15HT", "chance1x2_quote_diffRealCurr2", "goalNoGoal_comparison_affini", "goalNoGoal_multigoal_m24Home", "evaluation_valUnderOver", "underOver_flashback_under35", "chance1x2_bookkeeping_status", "goalNoGoal_bookkeeping_actual", "goalNoGoal_multigoal_m24Away", "chance1x2_flashback_pHtx", "chance1x2_bookkeeping_p1", "chance1x2_quote_initial1", "goalNoGoal_chance_noGoal", "underOver_chance_under05HT", "underOver_quote_realU", "underOver_chance_over052HT", "chance1x2_chance_px", "underOver_bookkeeping_status", "evaluation_valScala", "underOver_chance_over05HT", "underOver_quote_diffRealCurrU", "underOver_quote_realO", "underOver_comparison_flashback", "goalNoGoal_stats_avgGoalHome", "chance1x2_chance_p2Ht1x", "underOver_comparison_affini", "underOver_chance_under15HT", "chance1x2_flashback_pHt2", "goalNoGoal_multigoal_m13", "underOver_quote_diffInitialCurrU", "goalNoGoal_quote_diffRealCurrGG", "goalNoGoal_stats_avgGoalTakenHome", "goalNoGoal_quote_realNG", "goalNoGoal_quote_diffInitialCurrGG", "chance1x2_chance_p1", "underOver_quote_diffInitialCurrO", "chance1x2_quote_initial2", "chance1x2_quote_realx", "goalNoGoal_flashback_noGoal", "underOver_chance_over15", "goalNoGoal_bookkeeping_status", "chance1x2_quote_current1", "underOver_chance_under45", "chance1x2_quote_initialx", "goalNoGoal_bookkeeping_gg", "underOver_chance_over45", "chance1x2_comparison_flashback", "chance1x2_flashback_p2", "chance1x2_bookkeeping_px", "goalNoGoal_multigoal_m13Away", "chance1x2_chance_p2x", "chance1x2_quote_currentx", "chance1x2_bookkeeping_actual", "underOver_flashback_over25", "goalNoGoal_quote_currentNG", "chance1x2_quote_diffRealCurrx", "underOver_chance_under052HT", "underOver_bookkeeping_o", "underOver_chance_under35", "goalNoGoal_bookkeeping_avg", "chance1x2_quote_current2", "goalNoGoal_flashback_goal", "underOver_flashback_over05HT", "underOver_flashback_under05HT", "evaluation_valMetrica", "underOver_flashback_over15", "chance1x2_quote_real1", "goalNoGoal_quote_initialNG", "underOver_bookkeeping_actual", "goalNoGoal_quote_initialGG", "chance1x2_chance_p12", "underOver_quote_currentU", "underOver_flashback_under25", "evaluation_val1x2", "goalNoGoal_stats_avgGoalAway", "goalNoGoal_flashback_m35", "goalNoGoal_flashback_m24", "underOver_quote_initialU", "goalNoGoal_quote_diffRealCurrNG", "goalNoGoal_multigoal_m24", "underOver_bookkeeping_avg", "goalNoGoal_chance_odd", "chance1x2_flashback_p1", "goalNoGoal_flashback_m13", "underOver_flashback_over35", "goalNoGoal_multigoal_m13Home", "chance1x2_bookkeeping_p2", "underOver_quote_diffRealCurrO", "underOver_chance_under25", "goalNoGoal_quote_diffInitialCurrNG", "goalNoGoal_comparison_flashback", "underOver_quote_initialO", "chance1x2_chance_pHt2", "goalNoGoal_quote_currentGG", "chance1x2_bookkeeping_avg", "chance1x2_quote_diffInitialCurr2", "goalNoGoal_bookkeeping_ng", "underOver_flashback_under15"])]

In [7]:
from sklearn.compose import ColumnTransformer

transformers = datetime_transformers + bool_transformers + numerical_transformers

preprocessor = ColumnTransformer(transformers, remainder="passthrough", sparse_threshold=0)

## Train - Validation - Test Split
The input data is split by AutoML into 3 sets:
- Train (60% of the dataset used to train the model)
- Validation (20% of the dataset used to tune the hyperparameters of the model)
- Test (20% of the dataset used to report the true performance of the model on an unseen dataset)

`_automl_split_col_0000` contains the information of which set a given row belongs to.
We use this column to split the dataset into the above 3 sets. 
The column should not be used for training so it is dropped after split is done.

Given that `time` is provided as the `time_col`, the data is split based on time order,
where the most recent data is split to the test data.

In [8]:
# AutoML completed train - validation - test split internally and used _automl_split_col_0000 to specify the set
split_train_df = df_loaded.loc[df_loaded._automl_split_col_0000 == "train"]
split_val_df = df_loaded.loc[df_loaded._automl_split_col_0000 == "validate"]
split_test_df = df_loaded.loc[df_loaded._automl_split_col_0000 == "test"]

# Separate target column from features and drop _automl_split_col_0000
X_train = split_train_df.drop([target_col, "_automl_split_col_0000"], axis=1)
y_train = split_train_df[target_col]

X_val = split_val_df.drop([target_col, "_automl_split_col_0000"], axis=1)
y_val = split_val_df[target_col]

X_test = split_test_df.drop([target_col, "_automl_split_col_0000"], axis=1)
y_test = split_test_df[target_col]

## Train classification model
- Log relevant metrics to MLflow to track runs
- All the runs are logged under [this MLflow experiment](#mlflow/experiments/1125739752927671)
- Change the model parameters and re-run the training cell to log a different trial to the MLflow experiment
- To view the full list of tunable hyperparameters, check the output of the cell below

In [9]:
from sklearn.linear_model import LogisticRegression

help(LogisticRegression)

Help on class LogisticRegression in module sklearn.linear_model._logistic:

class LogisticRegression(sklearn.linear_model._base.LinearClassifierMixin, sklearn.linear_model._base.SparseCoefMixin, sklearn.base.BaseEstimator)
 |  LogisticRegression(penalty='l2', *, dual=False, tol=0.0001, C=1.0, fit_intercept=True, intercept_scaling=1, class_weight=None, random_state=None, solver='lbfgs', max_iter=100, multi_class='deprecated', verbose=0, warm_start=False, n_jobs=None, l1_ratio=None)
 |  
 |  Logistic Regression (aka logit, MaxEnt) classifier.
 |  
 |  This class implements regularized logistic regression using the
 |  'liblinear' library, 'newton-cg', 'sag', 'saga' and 'lbfgs' solvers. **Note
 |  that regularization is applied by default**. It can handle both dense
 |  and sparse input. Use C-ordered arrays or CSR matrices containing 64-bit
 |  floats for optimal performance; any other input format will be converted
 |  (and copied).
 |  
 |  The 'newton-cg', 'sag', and 'lbfgs' solvers s

### Define the objective function
The objective function used to find optimal hyperparameters. By default, this notebook only runs
this function once (`max_evals=1` in the `hyperopt.fmin` invocation) with fixed hyperparameters, but
hyperparameters can be tuned by modifying `space`, defined below. `hyperopt.fmin` will then use this
function's return value to search the space to minimize the loss.

In [10]:
from mlflow.models import Model, infer_signature, ModelSignature
from mlflow.pyfunc import PyFuncModel
from mlflow import pyfunc
import sklearn
from sklearn import set_config
from sklearn.pipeline import Pipeline

from hyperopt import hp, tpe, fmin, STATUS_OK, Trials

def objective(params):
  with mlflow.start_run() as mlflow_run:
    sklr_classifier = LogisticRegression(**params)

    model = Pipeline([
        ("column_selector", col_selector),
        ("preprocessor", preprocessor),
        ("classifier", sklr_classifier),
    ])

    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        silent=True)

    model.fit(X_train, y_train)

    
    # Log metrics for the training set
    mlflow_model = Model()
    pyfunc.add_to_model(mlflow_model, loader_module="mlflow.sklearn")
    pyfunc_model = PyFuncModel(model_meta=mlflow_model, model_impl=model)
    training_eval_result = mlflow.evaluate(
        model=pyfunc_model,
        data=X_train.assign(**{str(target_col):y_train}),
        targets=target_col,
        model_type="classifier",
        evaluator_config = {"log_model_explainability": False,
                            "metric_prefix": "training_" , "pos_label": 1 }
    )
    sklr_training_metrics = training_eval_result.metrics
    # Log metrics for the validation set
    val_eval_result = mlflow.evaluate(
        model=pyfunc_model,
        data=X_val.assign(**{str(target_col):y_val}),
        targets=target_col,
        model_type="classifier",
        evaluator_config = {"log_model_explainability": False,
                            "metric_prefix": "val_" , "pos_label": 1 }
    )
    sklr_val_metrics = val_eval_result.metrics
    # Log metrics for the test set
    test_eval_result = mlflow.evaluate(
        model=pyfunc_model,
        data=X_test.assign(**{str(target_col):y_test}),
        targets=target_col,
        model_type="classifier",
        evaluator_config = {"log_model_explainability": False,
                            "metric_prefix": "test_" , "pos_label": 1 }
    )
    sklr_test_metrics = test_eval_result.metrics

    loss = -sklr_val_metrics["val_roc_auc"]

    # Truncate metric key names so they can be displayed together
    sklr_val_metrics = {k.replace("val_", ""): v for k, v in sklr_val_metrics.items()}
    sklr_test_metrics = {k.replace("test_", ""): v for k, v in sklr_test_metrics.items()}

    return {
      "loss": loss,
      "status": STATUS_OK,
      "val_metrics": sklr_val_metrics,
      "test_metrics": sklr_test_metrics,
      "model": model,
      "run": mlflow_run,
    }

### Configure the hyperparameter search space
Configure the search space of parameters. Parameters below are all constant expressions but can be
modified to widen the search space. For example, when training a decision tree classifier, to allow
the maximum tree depth to be either 2 or 3, set the key of 'max_depth' to
`hp.choice('max_depth', [2, 3])`. Be sure to also increase `max_evals` in the `fmin` call below.

See https://docs.databricks.com/applications/machine-learning/automl-hyperparam-tuning/index.html
for more information on hyperparameter tuning as well as
http://hyperopt.github.io/hyperopt/getting-started/search_spaces/ for documentation on supported
search expressions.

For documentation on parameters used by the model in use, please see:
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

NOTE: The above URL points to a stable version of the documentation corresponding to the last
released version of the package. The documentation may differ slightly for the package version
used by this notebook.

In [11]:
space = {
  "C": 0.008107306487633793,
  "penalty": "l1",
  "solver": "saga",
  "random_state": 495629405,
}

### Run trials
When widening the search space and training multiple models, switch to `SparkTrials` to parallelize
training on Spark:
```
from hyperopt import SparkTrials
trials = SparkTrials()
```

NOTE: While `Trials` starts an MLFlow run for each set of hyperparameters, `SparkTrials` only starts
one top-level run; it will start a subrun for each set of hyperparameters.

See http://hyperopt.github.io/hyperopt/scaleout/spark/ for more info.

In [12]:
import pandas as pd

trials = Trials()
fmin(objective,
     space=space,
     algo=tpe.suggest,
     max_evals=1,  # Increase this when widening the hyperparameter search space.
     trials=trials)

best_result = trials.best_trial["result"]
model = best_result["model"]
mlflow_run = best_result["run"]

display(
  pd.DataFrame(
    [best_result["val_metrics"], best_result["test_metrics"]],
    index=["validation", "test"]))

set_config(display="diagram")
model

  0%|          | 0/1 [00:01<?, ?trial/s, best loss=?]

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

2026/01/25 11:29:04 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.

2026/01/25 11:29:04 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...

2026/01/25 11:29:08 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.

2026/01/25 11:29:08 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...

2026/01/25 11:29:12 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.

2026/01/25 11:29:12 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...



🏃 View run selective-lamb-197 at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/3256615592902883/runs/cc671ce14cef4dc8ba2c3fc2cb7387d9

🧪 View experiment at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/3256615592902883

100%|██████████| 1/1 [00:23<00:00, 23.88s/trial, best loss: -0.7264532600157108]


,score,true_negatives,false_positives,false_negatives,true_positives,example_count,accuracy_score,recall_score,precision_score,f1_score,log_loss,roc_auc,precision_recall_auc
validation,0.718686,24,128,9,326,487,0.718686,0.973134,0.718062,0.826362,0.555426,0.726453,0.848773
test,0.709611,32,130,12,315,489,0.709611,0.963303,0.707865,0.816062,0.587015,0.683062,0.812238


Pipeline(steps=[('column_selector',
                 ColumnSelector(cols=['goalNoGoal_stats_avgGoalTakenAway',
                                      'underOver_quote_currentO',
                                      'goalNoGoal_chance_goal',
                                      'chance1x2_chance_p2',
                                      'chance1x2_chance_pHtx',
                                      'chance1x2_chance_p2Ht2x',
                                      'chance1x2_comparison_affini',
                                      'underOver_chance_over35',
                                      'underOver_chance_under15',
                                      'chance1x2_flashback_px',
                                      'goalNoGoal_multigoal_m35'...
                                                   'chance1x2_quote_diffInitialCurr1',
                                                   'chance1x2_quote_diffRealCurr1',
                                                   'goalNoGoal_quote_realGG',
                                                   'chance1x2_flashback_pHt1',
                                                   'underOver_chance_over15HT',
                                                   'chance1x2_quote_diffRealCurr2',
                                                   'goalNoGoal_comparison_affini',
                                                   'goalNoGoal_multigoal_m24Home', ...])])),
                ('classifier',
                 LogisticRegression(C=0.008107306487633793, penalty='l1',
                                    random_state=495629405, solver='saga'))])

<Figure size 1050x700 with 0 Axes>

## Feature importance

SHAP is a game-theoretic approach to explain machine learning models, providing a summary plot
of the relationship between features and model output. Features are ranked in descending order of
importance, and impact/color describe the correlation between the feature and the target variable.
- Generating SHAP feature importance is a very memory intensive operation, so to ensure that AutoML can run trials without
  running out of memory, we disable SHAP by default.<br />
  You can set the flag defined below to `shap_enabled = True` and re-run this notebook to see the SHAP plots.
- To reduce the computational overhead of each trial, a single example is sampled from the validation set to explain.<br />
  For more thorough results, increase the sample size of explanations, or provide your own examples to explain.
- SHAP cannot explain models using data with nulls; if your dataset has any, both the background data and
  examples to explain will be imputed using the mode (most frequent values). This affects the computed
  SHAP values, as the imputed samples may not match the actual data distribution.

For more information on how to read Shapley values, see the [SHAP documentation](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html).

> **NOTE:** SHAP run may take a long time with the datetime columns in the dataset.

In [13]:
# Set this flag to True and re-run the notebook to see the SHAP plots
shap_enabled = False

In [14]:
if shap_enabled:
    mlflow.autolog(disable=True)
    mlflow.sklearn.autolog(disable=True)
    from shap import KernelExplainer, summary_plot
    # SHAP cannot explain models using data with nulls.
    # To enable SHAP to succeed, both the background data and examples to explain are imputed with the mode (most frequent values).
    mode = X_train.mode().iloc[0]

    # Sample background data for SHAP Explainer. Increase the sample size to reduce variance.
    train_sample = X_train.sample(n=min(100, X_train.shape[0]), random_state=495629405).fillna(mode)

    # Sample some rows from the validation set to explain. Increase the sample size for more thorough results.
    example = X_val.sample(n=min(100, X_val.shape[0]), random_state=495629405).fillna(mode)

    # Use Kernel SHAP to explain feature importance on the sampled rows from the validation set.
    predict = lambda x: model.predict(pd.DataFrame(x, columns=X_train.columns))
    explainer = KernelExplainer(predict, train_sample, link="identity")
    shap_values = explainer.shap_values(example, l1_reg=False, nsamples=500)
    summary_plot(shap_values, example, class_names=model.classes_)

## Inference
[The MLflow Model Registry](https://docs.databricks.com/applications/mlflow/model-registry.html) is a collaborative hub where teams can share ML models, work together from experimentation to online testing and production, integrate with approval and governance workflows, and monitor ML deployments and their performance. The snippets below show how to add the model trained in this notebook to the model registry and to retrieve it later for inference.

> **NOTE:** The `model_uri` for the model already trained in this notebook can be found in the cell below

### Register to Model Registry
```
model_name = "Example"

model_uri = f"runs:/{ mlflow_run.info.run_id }/model"
registered_model_version = mlflow.register_model(model_uri, model_name)
```

### Load from Model Registry
```
model_name = "Example"
model_version = registered_model_version.version

model_uri=f"models:/{model_name}/{model_version}"
model = mlflow.pyfunc.load_model(model_uri=model_uri)
model.predict(input_X)
```

### Load model without registering
```
model_uri = f"runs:/{ mlflow_run.info.run_id }/model"

model = mlflow.pyfunc.load_model(model_uri=model_uri)
model.predict(input_X)
```

## Confusion matrix, ROC and Precision-Recall curves for validation data

We show the confusion matrix, ROC and Precision-Recall curves of the model on the validation data.

For the plots evaluated on the training and the test data, check the artifacts on the MLflow run page.

In [15]:
import uuid
from IPython.display import Image

# Download the artifact
eval_path = mlflow.artifacts.download_artifacts(run_id=mlflow_run.info.run_id, dst_path='./artifacts')

### Confusion matrix for validation dataset

In [0]:
eval_confusion_matrix_path = os.path.join(eval_path, "val_confusion_matrix.png")
display(Image(filename=eval_confusion_matrix_path))

### ROC curve for validation dataset

In [0]:
eval_roc_curve_path = os.path.join(eval_path, "val_roc_curve_plot.png")
display(Image(filename=eval_roc_curve_path))

### Precision-Recall curve for validation dataset

In [0]:
eval_pr_curve_path = os.path.join(eval_path, "val_precision_recall_curve_plot.png")
display(Image(filename=eval_pr_curve_path))

# Expected Values (EVs)

In [16]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score

model_name = "bet_master_analytics.models.win_1x_logistic_regression"
model_version = "1"
odds_name = "chance1x2_quote_current1"

In [52]:
# def get_threshold_and_ev(X_val, y_val, model_name, model_version, odds_name, P_MIN = 0.75, N_MIN = 0.1):
#     model_uri=f"models:/{model_name}/{model_version}"
#     model = mlflow.sklearn.load_model(model_uri=model_uri)
#     # -----------------------------
#     # 1. Model raw predictions
#     # -----------------------------
#     p_model = model.predict_proba(X_val)[:, 1]
#
#     # -----------------------------
#     # 2. Tuning the probabilities
#     # -----------------------------
#     calibrator = CalibratedClassifierCV(
#         estimator=model,
#         method="isotonic",
#         cv="prefit"
#     )
#     calibrator.fit(X_val, y_val)
#
#     p_cal = calibrator.predict_proba(X_val)[:, 1]
#
#     df_val = pd.concat([X_val, y_val], axis=1).reset_index(drop=True)
#     df_val["p_cal"] = p_cal
#
#     # -----------------------------
#     # 3. Linkining the odds with the chosen event
#     # -----------------------------
#     df_val["odds"] = df_val[odds_name]
#
#     # -----------------------------
#     # 4. Calculate Expected Value
#     # -----------------------------
#     df_val["EV"] = df_val["p_cal"] * df_val["odds"] - 1
#
#     # -----------------------------
#     # 5. Sweep treshold
#     # -----------------------------
#     results = []
#
#     THRESHOLDS = np.linspace(0.55, 0.90, 71)
#
#     N_MIN = int(len(y_val) * N_MIN)
#     for t in THRESHOLDS:
#         sel = df_val[df_val["p_cal"] >= t].copy()
#         if len(sel) < N_MIN:
#             continue
#
#         sel["y"] = sel["win_1x"].astype(int)  # 1 se vince, 0 se perde
#
#         precision = precision_score(sel["y"], np.ones(len(sel)))  # precision rispetto a tutti predetti
#         mean_ev = sel["EV"].mean()
#
#         results.append({
#             "threshold": t,
#             "precision": precision,
#             "mean_EV": mean_ev,
#             "n_bets": len(sel)
#         })
#
#     res_df = pd.DataFrame(results)
#
#     # -----------------------------
#     # 6. Final treshold (massimo EV con vincoli)
#     # -----------------------------
#     optimal = (
#         res_df[res_df["precision"] >= P_MIN]
#         .sort_values("mean_EV", ascending=False)
#         .head(1)
#     )
#
#     print(optimal)
#     return optimal
#
# optimal = get_threshold_and_ev(X_val, y_val, model_name, model_version, odds_name, N_MIN=0.9)

KeyError: 'precision'

In [85]:
# def get_threshold_and_ev(
#     df_val, model_name, model_version, odds_name,
#     P_MIN=0.75, N_MIN=0.1, EV_MIN=0.1, EV_MAX=0.5,
#     weight_ev=0.7, weight_nbets=0.3
# ):
#     """
#     Calcola la soglia ottimale di un classificatore per scommesse
#     massimizzando un punteggio combinato tra EV vicino al centro e numero di partite.
#
#     weight_ev + weight_nbets deve essere 1.0
#     """
#     assert abs(weight_ev + weight_nbets - 1.0) < 1e-6, "I pesi devono sommare a 1"
#
#     model_uri = f"models:/{model_name}/{model_version}"
#     model = mlflow.sklearn.load_model(model_uri=model_uri)
#
#     # -----------------------------
#     # 1. Model raw predictions
#     # -----------------------------
#     p_model = model.predict_proba(X_val)[:, 1]
#
#     # -----------------------------
#     # 2. Calibration
#     # -----------------------------
#     calibrator = CalibratedClassifierCV(estimator=model, method="isotonic", cv="prefit")
#     calibrator.fit(X_val, y_val)
#     p_cal = calibrator.predict_proba(X_val)[:, 1]
#
#
#     df_val["p_cal"] = p_cal
#     df_val["odds"] = df_val[odds_name]
#
#     # -----------------------------
#     # 3. Calculate Expected Value
#     # -----------------------------
#     df_val["EV"] = df_val["p_cal"] * df_val["odds"] - 1
#
#     # -----------------------------
#     # 4. Sweep thresholds
#     # -----------------------------
#     results = []
#     THRESHOLDS = np.linspace(0.55, 0.90, 71)
#     N_MIN = int(len(y_val) * N_MIN)
#
#     for t in THRESHOLDS:
#         sel = df_val[df_val["p_cal"] >= t].copy()
#         if len(sel) < N_MIN:
#             continue
#
#         sel["y"] = sel["win_1x"].astype(int)
#         precision = precision_score(sel["y"], np.ones(len(sel)))
#         mean_ev = sel["EV"].mean()
#
#         results.append({
#             "threshold": t,
#             "precision": precision,
#             "mean_EV": mean_ev,
#             "n_bets": len(sel),
#             "perc_bets": int(len(sel)/len(y_val) * 100)
#         })
#
#     res_df = pd.DataFrame(results)
#
#     # -----------------------------
#     # 5. Compute weighted robust score
#     # -----------------------------
#     EV_CENTER = (EV_MIN + EV_MAX) / 2
#
#     res_df = res_df[
#         (res_df["precision"] >= P_MIN) &
#         (res_df["mean_EV"] >= EV_MIN) &
#         (res_df["mean_EV"] <= EV_MAX)
#     ].copy()
#
#     if res_df.empty:
#         print("Nessuna soglia valida trovata!")
#         return None
#
#     # Score combinato pesato
#     res_df["score"] = (
#         weight_nbets * (res_df["n_bets"] / res_df["n_bets"].max()) +
#         weight_ev * (1 - abs(res_df["mean_EV"] - EV_CENTER) / (EV_MAX - EV_MIN))
#     )
#
#     optimal = res_df.sort_values("score", ascending=False).head(1)
#     print(optimal)
#
#     # optimal = {x: optimal[x] for x in optimal}
#     return optimal.iloc[0].to_dict()
#
# optimal = get_threshold_and_ev(
#     X_val, y_val, model_name, model_version, odds_name,
#     P_MIN=0.75, N_MIN=0, EV_MIN=0.1, EV_MAX=0.5,
#     weight_ev=0.8, weight_nbets=0.2
# )


    threshold  precision   mean_EV  n_bets  perc_bets     score
61      0.855   0.928571  0.276758      84         17  0.800182


In [26]:
def get_threshold_and_test_ev(
    X_val, y_val,
    X_test, y_test,
    model_name, model_version, odds_name,
    P_MIN=0.75, N_MIN_PERC=0.1,
    EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.7, weight_nbets=0.3
):
    assert abs(weight_ev + weight_nbets - 1.0) < 1e-6

    # -----------------------------
    # 1. Load model
    # -----------------------------
    model_uri = f"models:/{model_name}/{model_version}"
    model = mlflow.sklearn.load_model(model_uri)

    # -----------------------------
    # 2. Calibration (fit on VAL)
    # -----------------------------
    calibrator = CalibratedClassifierCV(
        estimator=model,
        method="isotonic",
        cv="prefit"
    )
    calibrator.fit(X_val, y_val)

    # -----------------------------
    # 3. Predict on VAL
    # -----------------------------
    p_val = calibrator.predict_proba(X_val)[:, 1]
    df_val = pd.concat([X_val, y_val], axis=1).reset_index(drop=True)
    df_val["p_cal"] = p_val
    df_val["odds"] = df_val[odds_name]
    df_val["EV"] = df_val["p_cal"] * df_val["odds"] - 1

    # -----------------------------
    # 4. Sweep thresholds (VAL)
    # -----------------------------
    results = []
    THRESHOLDS = np.linspace(0.55, 0.90, 71)
    N_MIN = int(len(y_val) * N_MIN_PERC)

    for t in THRESHOLDS:
        sel = df_val[df_val["p_cal"] >= t]
        if len(sel) < N_MIN:
            continue

        precision = precision_score(
            sel["win_1x"].astype(int),
            np.ones(len(sel))
        )
        mean_ev = sel["EV"].mean()

        if precision < P_MIN or not (EV_MIN <= mean_ev <= EV_MAX):
            continue

        results.append({
            "threshold": t,
            "mean_EV_val": mean_ev,
            "precision_val": precision,
            "n_bets_val": len(sel),
            "n_bets_val_perc": int(len(sel) / len(y_val) * 100),
        })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        print("Nessuna soglia valida trovata")
        return None

    # -----------------------------
    # 5. Score robusto (VAL)
    # -----------------------------
    EV_CENTER = (EV_MIN + EV_MAX) / 2

    res_df["score"] = (
        weight_nbets * (res_df["n_bets_val"] / res_df["n_bets_val"].max()) +
        weight_ev * (
            1 - abs(res_df["mean_EV_val"] - EV_CENTER) / (EV_MAX - EV_MIN)
        )
    )

    optimal = res_df.sort_values("score", ascending=False).iloc[0]
    t_star = optimal["threshold"]

    # -----------------------------
    # 6. FINAL EVALUATION ON TEST
    # -----------------------------
    p_test = calibrator.predict_proba(X_test)[:, 1]
    df_test = pd.concat([X_test, y_test], axis=1).reset_index(drop=True)
    df_test["p_cal"] = p_test
    df_test["odds"] = df_test[odds_name]
    df_test["EV"] = df_test["p_cal"] * df_test["odds"] - 1

    sel_test = df_test[df_test["p_cal"] >= t_star]

    test_metrics = {
        "threshold": t_star,
        "n_bets_test_perc": int(len(sel_test) / len(y_test) * 100),
        "mean_EV_test": sel_test["EV"].mean() if len(sel_test) > 0 else np.nan,
        "ROI_test": sel_test["EV"].sum() / len(sel_test) if len(sel_test) > 0 else np.nan,
        "precision_test": precision_score(
            sel_test["win_1x"].astype(int),
            np.ones(len(sel_test))
        ) if len(sel_test) > 0 else np.nan
    }

    return {
        "val": optimal.to_dict(),
        "test": test_metrics
    }


optimal = get_threshold_and_test_ev(
    X_val, y_val, X_test, y_test, model_name, model_version, odds_name,
    P_MIN=0.75, EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.8, weight_nbets=0.2
)

Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

🏃 View run languid-rat-698 at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/3256615592902883/runs/53df6e8cb30f4b1387f28590d4343366
🧪 View experiment at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/3256615592902883


In [28]:
optimal

{'val': {'threshold': 0.865,
  'mean_EV_val': 0.2767575187969925,
  'precision_val': 0.9285714285714286,
  'n_bets_val': 84.0,
  'n_bets_val_perc': 17.0,
  'score': 0.8001817042606517},
 'test': {'threshold': np.float64(0.865),
  'n_bets_test_perc': 18,
  'mean_EV_test': np.float64(0.2822611132133517),
  'ROI_test': np.float64(0.2822611132133517),
  'precision_test': 0.8791208791208791}}

In [23]:
from mlflow.tracking import MlflowClient
import json

def register_model(bet, model, **kwargs):
    # Register Model
    model_name = f"bet_master_analytics.models.{bet}_{model}"
    model_uri = f"runs:/{mlflow_run.info.run_id}/model"

    client = MlflowClient()

    tags = {
        "bet": bet,
        "model": model,
        "env": "dev",
        #"mlflow.run_id": mlflow_run.info.run_id,
    } | kwargs

    registered_model_version = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    for key, value in tags.items():
        client.set_model_version_tag(
            name=registered_model_version.name,
            version=registered_model_version.version,
            key=key,
            value=value
        )

    # model_uri for the generated model
    print(f"runs:/{mlflow_run.info.run_id}/model")

register_model(bet="win_1x", model="logistic_regression", **optimal['test'])

Registered model 'bet_master_analytics.models.win_1x_logistic_regression' already exists. Creating a new version of this model...
2026/01/25 11:38:31 WARNING mlflow.tracking._model_registry.fluent: Run with id cc671ce14cef4dc8ba2c3fc2cb7387d9 has no artifacts at artifact path 'model', registering model based on models:/m-61277744553d41d0884ae054f2633c4f instead


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Created version '6' of model 'bet_master_analytics.models.win_1x_logistic_regression'.


runs:/cc671ce14cef4dc8ba2c3fc2cb7387d9/model


In [ ]:
# todo: retrain
# todo: definire soglia su val set, testare soglia su test set

# Save the model

In [0]:
import joblib

class BettingModelWrapper:
    def __init__(self, model, threshold):
        self.model = model
        self.threshold = threshold
        
    def predict(self, X):
        p = self.model.predict_proba(X)[:, 1]
        return (p >= self.threshold).astype(int)
    
    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

# Crea wrapper
wrapper = BettingModelWrapper(model, threshold=0.6)

joblib.dump(wrapper, "artifacts/model_thresholded.pkl")
 
